# 02 — Data Preprocessing

In [1]:
import sys
sys.path.insert(0, "..")
from src import config, data_loader, preprocessing

config.ensure_directories()
df_raw = data_loader.load_raw_dataset()
print("Raw shape:", df_raw.shape)


2026-09-01 22:00:43,653 | INFO     | src.data_loader | Loaded raw dataset: 583 rows, 24 columns from E:\Job Base Programe\ResearchMind\data\raw\research_papers_raw.csv


Raw shape: (583, 24)


## Stage 1 — Cleaning

Removes null rows, malformed records, and duplicates; normalizes titles, abstracts, authors, and years; validates URLs; builds the combined `search_text` field.

In [2]:
df_clean = preprocessing.clean_dataset(df_raw)
print("Clean shape:", df_clean.shape)
df_clean[["title", "search_text"]].head(3)


2026-09-01 22:00:43,703 | INFO     | src.preprocessing | remove_null_rows: 583 -> 583 rows
2026-09-01 22:00:44,115 | INFO     | src.preprocessing | remove_malformed_records: 583 -> 583 rows
2026-09-01 22:00:44,216 | INFO     | src.preprocessing | deduplicate_papers: 583 -> 583 rows
2026-09-01 22:00:44,232 | INFO     | src.preprocessing | remove_without_useful_text: 583 -> 583 rows


Clean shape: (583, 26)


,title,search_text
0,Comparing Physician and Artificial Intelligenc...,Comparing Physician and Artificial Intelligenc...
1,Foundation models for generalist medical artif...,Foundation models for generalist medical artif...
2,Artificial Intelligence in Education: A Review,Artificial Intelligence in Education: A Review...


In [3]:
data_loader.save_dataframe(df_clean, config.CLEAN_CSV_PATH)


2026-09-01 22:00:44,750 | INFO     | src.data_loader | Saved 583 rows to E:\Job Base Programe\ResearchMind\data\processed\research_papers_clean.csv


## Stage 2 — Final quality filtering

Removes any remaining low-quality records (very short titles/abstracts).

In [4]:
df_final = preprocessing.quality_filter(df_clean)
print("Final shape:", df_final.shape)
data_loader.save_dataframe(df_final, config.FINAL_CSV_PATH)


2026-09-01 22:00:44,862 | INFO     | src.preprocessing | quality_filter: 583 -> 583 rows
2026-09-01 22:00:44,930 | INFO     | src.data_loader | Saved 583 rows to E:\Job Base Programe\ResearchMind\data\processed\research_papers_final.csv


Final shape: (583, 26)


## Validation report

In [5]:
import json
report = preprocessing.validate_dataset(df_final)
print(json.dumps(report, indent=2, default=str))
assert report["num_rows"] > 0, "Final dataset is empty — check upstream filtering thresholds."


{
  "num_rows": 583,
  "num_columns": 26,
  "columns_present": {
    "paper_id": "paper_id",
    "title": "title",
    "abstract": "abstract",
    "authors": "authors",
    "year": "year",
    "venue": "venue",
    "url": "url",
    "citation_count": "citation_count",
    "reference_count": "reference_count",
    "publication_date": "publication_date",
    "fields_of_study": "fields_of_study",
    "influential_citation_count": "influential_citation_count",
    "tldr": "tldr",
    "research_topic": "research_topic"
  },
  "columns_missing": [],
  "has_search_text": true,
  "duplicate_titles": 0
}
